<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 220px; height: 150px; vertical-align: middle;">
            <img src="../assets/aaa.png" width="220" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Autonomous Traders</h2>
            <span style="color:#ff7800;">An equity trading simulation to illustrate autonomous agents powered by tools and resources from MCP servers.
            </span>
        </td>
    </tr>
</table>

### Week 6 Day 4

Welcome to the Capstone project!

# Autonomous Traders

An equity trading simulation with four Traders and a Researcher, powered by a team of MCP servers and their tools and resources:

1. Our home-made Accounts server, written by our engineering team
2. Push notifications, to alert us when something happens
3. Market data, for live or simulated share prices
4. Fetch, to read a web page through a local headless browser
5. Tavily, for web search
6. Memory, a knowledge graph the researcher writes to and reads back

Plus resources to read each trader's account and their investment strategy.

The whole system already lives in the `backend` package. Today we tour the key pieces, build and run a single trader to watch it work, then bring up a dashboard to see the full team trading.

## The architecture

Here is how the pieces fit together. Each of the four traders is an agent with its own MCP servers for accounts, push notifications and market data, and it calls a researcher agent as a tool. The researcher is itself an agent, with its own MCP servers for fetching pages, web search and memory. Orange is an agent, blue is an MCP server, and there are six MCP servers in all.

While I've given the Agents names like "Trader" and "Researcher", my motivation for this architecture was managing the context effectively with the right prompts and tools for reliable outcomes. It's key to avoid the trap of assigning Agent responsibilities just because that's how human teams are organized..

<img src="../assets/architecture.png" width="820" />

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">One more time --</h2>
            <span style="color:#ff7800;">Please do not use this for actual trading decisions!!
            </span>
        </td>
    </tr>
</table>

In [1]:
import sys, os

if sys.stderr is None or not hasattr(sys.stderr, "fileno"):
    sys.stderr = open(os.devnull, "w")
else:
    try:
        sys.stderr.fileno()
    except Exception:
        sys.stderr = open(os.devnull, "w")

import functools
import subprocess
import mcp.client.stdio

mcp.client.stdio.stdio_client = functools.partial(
    mcp.client.stdio.stdio_client, errlog=subprocess.DEVNULL
)


In [2]:
from dotenv import load_dotenv
import json
from contextlib import AsyncExitStack
from agents import Runner, trace, add_trace_processor
from IPython.display import Markdown, display
from backend.market import get_share_price
from backend.accounts import Account
from backend.accounts_client import read_accounts_resource
from backend.reset import reset_traders
from backend.mcp_servers import trader_mcp_servers, researcher_mcp_servers
from backend.traders import get_researcher, get_researcher_tool, Trader
from backend.tracers import LogTracer

load_dotenv(override=True)

True

### A quick note for Windows

Launching a local MCP server from a notebook hits one rough edge on Windows. The server writes its startup output to stderr, but inside a Windows Jupyter kernel that stream has no real file handle behind it, so the launch fails with an `io.UnsupportedOperation: fileno` error, while Mac and Linux are unaffected.

The fix is to send the server's stderr to the null device, so it always has somewhere real to write. We do that once in the next cell, which lets every MCP server below start cleanly. On Mac and Linux it costs nothing beyond keeping the server's startup banner out of the notebook.

In [3]:
# On Windows, a stdio MCP server started from a Jupyter kernel writes to a stderr stream with no
# real file descriptor and crashes with io.UnsupportedOperation: fileno. We send the server's
# stderr to the null device so it always has somewhere real to write. Mac and Linux are unaffected.
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

## A tour of the backend

The trading floor is already built as a Python package in `backend`. Before we run it, let's look at a few of the pieces it gives us.

### Market data

`get_share_price` returns the latest price for a symbol. With a Massive API key it uses live market data; without one it falls back to a simulator, so the floor runs either way.

In [4]:
get_share_price("AAPL")

185.88

### The traders' accounts

Each trader has an account: a balance, holdings, a transaction history and an investment strategy. The `Account` model is the heart of our home-made Accounts server. Let's reset the four traders to their starting strategies and look at one of them.

#### Ed note:

I have commented out the first line because I don't want to reset my traders!

In [5]:
# reset_traders() 
warren = Account.get("Warren")
print("Balance:", warren.balance)
print("Strategy:", warren.get_strategy())

Balance: 10000.0
Strategy: 
You are Warren, and you are named in homage to your role model, Warren Buffett.
You are a value-oriented investor who prioritizes long-term wealth creation.
You identify high-quality companies trading below their intrinsic value.
You invest patiently and hold positions through market fluctuations, 
relying on meticulous fundamental analysis, steady cash flows, strong management teams, 
and competitive advantages. You rarely react to short-term market movements, 
trusting your deep research and value-driven strategy.



### The MCP servers, and their tools

The trader and the researcher each get their own set of MCP servers, and `mcp_servers.py` builds them for us. Let's start each one and count the tools they expose.

In [6]:
traders = trader_mcp_servers()
researchers = researcher_mcp_servers("Warren")

print("TRADER SERVERS")
for i, server in enumerate(traders):
    try:
        async with server:
            tools = await server.list_tools()
            print(f"Trader {i}: ✅ {len(tools)} tools")
    except Exception as e:
        print(f"Trader {i}: ❌ {type(e).__name__}: {e}")

print("\nRESEARCHER SERVERS")
for i, server in enumerate(researchers):
    try:
        async with server:
            tools = await server.list_tools()
            print(f"Researcher {i}: ✅ {len(tools)} tools")
    except Exception as e:
        print(f"Researcher {i}: ❌ {type(e).__name__}: {e}")


TRADER SERVERS
Trader 0: ✅ 5 tools
Trader 1: ✅ 1 tools
Trader 2: ✅ 1 tools

RESEARCHER SERVERS
Researcher 0: ✅ 1 tools
Researcher 1: ✅ 1 tools
Researcher 2: ✅ 6 tools


## The Researcher

The trader does not search the web itself. Instead it calls a Researcher agent as one of its tools. The researcher has its own MCP servers: Fetch to read pages, Tavily for web search, and a Memory it writes to and reads back.

Tavily offers several tools, from plain search to a heavyweight deep-research mode. We restrict its server to `tavily_search`, which keeps the researcher fast and focused. Choosing which tools an agent can see is itself context engineering.

Wrapping an agent as a tool is different from a handoff: with a tool the trader stays in control and gets the researcher's answer back, where a handoff would pass the whole conversation over.

In [7]:
async with AsyncExitStack() as stack:
    servers = [await stack.enter_async_context(server) for server in researcher_mcp_servers("Warren")]
    researcher = await get_researcher(servers, "gpt-5.6-sol")
    with trace("Researcher"):
        result = await Runner.run(researcher, "What's the latest news on Amazon?", max_turns=30)
display(Markdown(result.final_output))

## Amazon: latest developments as of August 8, 2026

**1. AWS-powered earnings beat drove the stock to record highs**  
Amazon’s second-quarter sales rose **20% to $200.6 billion**, while operating income increased **43% to $27.5 billion**. AWS was the standout:

- AWS revenue: approximately **$42.2 billion**, up **37%**
- Fastest AWS growth in 18 quarters
- AWS operating income: **$16.6 billion**, up from $10.2 billion

Amazon subsequently crossed a **$3 trillion market capitalization** for the first time on August 3.  
One caveat: reported net income of $62.6 billion included a **$53.4 billion pre-tax non-operating gain**, primarily related to Amazon’s Anthropic investment, so headline EPS overstates underlying operating earnings.

**2. AI spending remains both the catalyst and principal risk**  
Amazon reportedly increased planned 2026 capital expenditure to roughly **$220 billion**, primarily for AWS data centers, AI infrastructure and custom chips. Management says AI demand remains ahead of available capacity.

That supports AWS growth, but the investment cycle is pressuring free cash flow and will eventually produce substantially higher depreciation. Investors now need AWS growth and margins to remain strong enough to justify the spending.

**3. Guidance remains constructive**  
For Q3 2026, Amazon expects:

- Revenue of **$197–202 billion**, representing 9%–12% growth
- Operating income of **$22.5–26.5 billion**

Amazon noted that the timing of Prime Day depresses the reported growth comparison; excluding that timing effect, growth would be almost four percentage points higher.

**4. Zoox is starting commercial robotaxi service**  
Amazon-owned Zoox will begin charging for robotaxi rides in Las Vegas on **August 10**, its first commercial market. This follows federal clearance for its purpose-built vehicles, which lack steering wheels and pedals.

This is strategically significant as a validation milestone, although Zoox is unlikely to contribute meaningfully to Amazon’s near-term earnings.

**5. Amazon Pharmacy is expanding into Medicare weight-loss drugs**  
Eligible Medicare Part D patients can now obtain selected GLP-1 weight-loss medications through Amazon Pharmacy for **$50 per month**, with free home delivery and no Prime membership requirement. Same-day delivery is available in more than 3,100 U.S. communities.

The immediate financial effect should be small, but it strengthens the integration between Amazon Pharmacy and One Medical in a high-growth healthcare category.

**6. New legal and competitive pressures**

- A federal appeals court overturned a preliminary order that had blocked **Perplexity’s AI shopping agent** from accessing Amazon. The underlying case continues, but the decision could weaken Amazon’s control over shopping traffic and customer data as agentic commerce grows.
- **New Jersey sued Amazon**, alleging its Delivery Service Partner structure suppresses driver wages and competition. Amazon denies the allegations.
- Universal Electronics filed a patent lawsuit involving Amazon’s Echo, Fire TV and other smart-home products.

**7. Bezos filed to sell approximately $4.1 billion of stock**  
Jeff Bezos filed to sell **15 million Amazon shares** under a prearranged Rule 10b5-1 plan. The timing—immediately after the record high—generated attention, but the sale represents a relatively small portion of his holdings and is not necessarily a negative operating signal.

### Investor takeaway

The news flow is **fundamentally bullish but valuation-sensitive**. AWS acceleration shows that Amazon’s AI investments are producing tangible revenue and margin growth. The main questions are whether that growth can remain near current levels and whether returns on the enormous infrastructure budget will outweigh free-cash-flow and depreciation pressure. Regulatory cases and AI shopping agents are meaningful longer-term risks, but currently secondary to the AWS/capex story.

**Sources:** [Amazon Q2 results](https://ir.aboutamazon.com/news-release/news-release-details/2026/Amazon-com-Announces-Second-Quarter-Results) · [Reuters on $3 trillion valuation](https://www.reuters.com/legal/transactional/amazon-enters-3-trillion-club-ai-optimism-sweeps-through-wall-street-2026-08-03) · [Reuters on Perplexity ruling](https://www.reuters.com/business/retail-consumer/amazon-loses-us-court-ban-perplexitys-ai-shopping-tools-2026-08-04) · [Reuters on New Jersey lawsuit](https://www.reuters.com/world/new-jersey-ag-files-antitrust-lawsuit-against-amazon-2026-08-04) · [Amazon Pharmacy announcement](https://www.aboutamazon.com/news/retail/amazon-pharmacy-glp-1-medicare) · [CNBC on Zoox](https://www.cnbc.com/2026/08/05/amazon-zoox-paid-robotaxi-rides-las-vegas.html)

### Look at the trace

https://platform.openai.com/traces

### Wrapping the researcher as a tool

The trader does not talk to the researcher directly. We turn the whole researcher agent into a tool, and the trader calls it like any other tool. `get_researcher_tool` does this with `researcher.as_tool(...)`.

In [8]:
researcher_tool = await get_researcher_tool(
    researcher_mcp_servers("Warren"), "gpt-5.6-sol"
)
print("Tool name:", researcher_tool.name)
print("Description:", researcher_tool.description)

Tool name: Researcher
Description: This tool researches online for news and opportunities, either based on your specific request to look into a certain stock, or generally for notable financial news and opportunities. Describe what kind of research you're looking for.


## The Trader

The `Trader` class in `traders.py` brings it together. It builds the researcher as a tool, creates the trader agent over the trader's MCP servers (accounts, push and market data) with that tool, and runs it against a message built from the trader's strategy and current account.

There is one more piece worth seeing. The OpenAI Agents SDK lets you plug into its tracing, so you can follow what each agent does in code. `tracers.py` has a custom trace processor that records each trader's steps to the database, which is how we surface their thinking on the dashboard. We register it now, then run Warren.

In [9]:
add_trace_processor(LogTracer())
warren = Trader("Warren", "Patience", "gpt-5.6-sol")
await warren.run()

### How did Warren do?

Reading the account back through its MCP resource shows the trades Warren made and the state of the portfolio.

In [10]:
resources = await read_accounts_resource("Warren")
info = json.loads(resources)
print(info["transactions"][-1])

{'symbol': 'UNH', 'quantity': 10, 'price': 121.64280000000001, 'timestamp': '2026-08-08 11:43:31', 'rationale': 'Modest recovery position in a scaled, integrated healthcare franchise with durable data, provider, and distribution advantages; the verified $121.29 quote offers substantial upside to normalized earnings value, while small sizing recognizes regulatory, medical-cost, and execution uncertainty.'}


## The whole team, on a loop

`trading_floor.py` is the full backend implementation. It creates all four traders and runs them on a timer:

```
while True:
    await asyncio.gather(*[trader.run() for trader in traders])
    await asyncio.sleep(RUN_EVERY_N_MINUTES * 60)
```

A few optional settings in your `.env` file control it:

`RUN_EVERY_N_MINUTES=60` sets how often the team runs, defaulting to every 60 minutes.

`RUN_EVEN_WHEN_MARKET_IS_CLOSED=False` decides whether the traders run out of hours.

`USE_MANY_MODELS=False` runs all four traders on gpt-5.4-mini by default. Set it to true to give each trader a different model: GPT-5.5, DeepSeek V4, Gemini 3.5 Flash and Grok 4.3.

## The dashboard

Now let's watch it. The dashboard lives in the `demo` package, and `app.py` launches it.

Open a new terminal (the plus on the terminal panel), change to this directory and run the dashboard:

`cd 6_mcp`

`uv run app.py`

Then, to set the team trading, open another terminal, change to the same directory and start the engine:

`cd 6_mcp`

`uv run -m backend.trading_floor`

Watch the dashboard, and see your trading team in action.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Keep an eye on your API usage</h2>
            <span style="color:#ff7800;">The engine runs on a loop, every hour or however you set it. Watch your API usage and stop it when you have seen enough. I find myself watching this happily for hours, and I hope you will too.
            </span>
        </td>
    </tr>
</table>

## Almost there

You have an autonomous trading floor: four traders, a researcher, six MCP servers, persistent memory and a live dashboard.

In the final lab we give it a production frontend, a separate web app that talks to the same backend.